# Coffee Standard v8 — dataset identity and ontology audit
Tidak melakukan training. Audit seluruh split hanya untuk provenance, label, distribusi, dan leakage; tidak menjalankan evaluasi model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, os, shutil, subprocess, sys, tarfile
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-igem-paired-confirmation'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/coffee-detection-with-standard-v8-yolov8.tar',))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/coffee-detection-with-standard-v8-yolov8.tar')
DATA_ROOT=Path('/content/coffee-standard-v8-audit')
if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
with tarfile.open(ARCHIVE,'r') as archive: archive.extractall(DATA_ROOT,filter='data')
OUTPUT=PROJECT_ROOT/'evidence/coffee-detection-with-standard-v8/dataset_audit.json'
print('PROJECT:',PROJECT_ROOT); print('ARCHIVE:',ARCHIVE); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.analysis.coffee_standard_dataset_audit import audit_dataset
result=audit_dataset(DATA_ROOT,OUTPUT)
print('DECISION:',result['decision'])
print('IMAGES:',result['total_images'],'INSTANCES:',result['total_instances'])
print('EXACT CROSS-SPLIT:',result['exact_cross_split_groups'])
print('PARENT CROSS-SPLIT:',result['roboflow_parent_cross_split_groups'])
print('INVALID LABELS:',result['invalid_label_rows'])
print('DIRECT SNI21 MAPPING:',result['direct_sni21_mapping_count'],'/ 25')
print('AMBIGUOUS:',result['ambiguous_or_external_classes'])
print('SUMMARY:',OUTPUT)
print('Kirim output ini. Jangan training.')


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame(result['class_distribution']))
display(pd.DataFrame([{'split':k,**v} for k,v in result['splits'].items()]))
if result['parent_cross_split_examples']:
    display(pd.DataFrame([{k:v for k,v in row.items() if k!='members'} for row in result['parent_cross_split_examples'][:20]]))
